In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from scipy import stats
from src.data_loader import DataLoader

# Load cleaned data
loader = DataLoader(data_dir="../data")
df = pd.read_csv(loader.processed_dir / "cleaned_data.csv")

In [2]:
# 1. Comprehensive Summary Statistics
def comprehensive_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate comprehensive statistics for all numeric columns.
    """
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    
    stats_dict = {
        'Column': [], 'Mean': [], 'Median': [], 'Mode': [], 'Std': [],
        'Variance': [], 'Skewness': [], 'Kurtosis': [], 'Range': [],
        'IQR': [], 'Q1': [], 'Q3': [], 'CV': [], 'Missing %': []
    }
    
    for col in numeric_cols:
        data = df[col].dropna()
        stats_dict['Column'].append(col)
        stats_dict['Mean'].append(data.mean())
        stats_dict['Median'].append(data.median())
        stats_dict['Mode'].append(data.mode()[0] if not data.mode().empty else np.nan)
        stats_dict['Std'].append(data.std())
        stats_dict['Variance'].append(data.var())
        stats_dict['Skewness'].append(data.skew())
        stats_dict['Kurtosis'].append(data.kurtosis())
        stats_dict['Range'].append(data.max() - data.min())
        stats_dict['IQR'].append(data.quantile(0.75) - data.quantile(0.25))
        stats_dict['Q1'].append(data.quantile(0.25))
        stats_dict['Q3'].append(data.quantile(0.75))
        stats_dict['CV'].append(data.std() / data.mean() if data.mean() != 0 else np.nan)
        stats_dict['Missing %'].append(df[col].isnull().mean() * 100)
    
    return pd.DataFrame(stats_dict)

# Generate stats
stats_df = comprehensive_stats(df)
stats_df.to_csv('../reports/summary_statistics.csv', index=False)
print("Summary Statistics:")
print(stats_df.to_string())

Summary Statistics:
              Column       Mean  Median   Mode       Std   Variance  Skewness  Kurtosis  Range   IQR     Q1     Q3        CV  Missing %
0  sepal length (cm)   5.843624    5.80   5.00  0.830851   0.690314  0.312826 -0.569006   3.60  1.30   5.10   6.40  0.142181        0.0
1   sepal width (cm)   3.056376    3.00   3.00  0.425825   0.181327  0.182187 -0.164193   2.00  0.50   2.80   3.30  0.139324        0.0
2  petal length (cm)   3.748993    4.30   1.40  1.767791   3.125083 -0.263101 -1.408270   5.90  3.50   1.60   5.10  0.471537        0.0
3   petal width (cm)   1.194631    1.30   0.20  0.762622   0.581593 -0.090076 -1.339953   2.40  1.50   0.30   1.80  0.638375        0.0
4         sepal_area  17.818389   17.68  13.20  3.329560  11.085969  0.466877  0.978681  19.77  4.74  15.66  20.40  0.186861        0.0
5         petal_area   5.767919    5.59   0.28  4.717353  22.253419  0.281047 -1.141010  15.76  9.27   0.42   9.69  0.817860        0.0


In [3]:
# 2. Distribution Tests
def test_normality(df: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """
    Perform Shapiro-Wilk test for normality on numeric columns.
    """
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    
    results = {
        'Column': [], 'Statistic': [], 'P-value': [], 'Normal?': [], 'Interpretation': []
    }
    
    for col in numeric_cols:
        data = df[col].dropna()
        if len(data) > 3:
            stat, p_value = stats.shapiro(data)
            is_normal = p_value > alpha
            results['Column'].append(col)
            results['Statistic'].append(stat)
            results['P-value'].append(p_value)
            results['Normal?'].append(is_normal)
            results['Interpretation'].append("Normal" if is_normal else "Not normal")
    
    return pd.DataFrame(results)

normality_df = test_normality(df)
normality_df.to_csv('../reports/normality_tests.csv', index=False)
print("\nNormality Tests:")
print(normality_df.to_string())


Normality Tests:
              Column  Statistic       P-value  Normal? Interpretation
0  sepal length (cm)   0.975553  9.233521e-03    False     Not normal
1   sepal width (cm)   0.983427  7.008545e-02     True         Normal
2  petal length (cm)   0.876789  8.635087e-10    False     Not normal
3   petal width (cm)   0.901933  1.852986e-08    False     Not normal
4         sepal_area   0.977143  1.377211e-02    False     Not normal
5         petal_area   0.904693  2.669507e-08    False     Not normal


In [4]:
# 3. Correlation Analysis
def correlation_analysis(df: pd.DataFrame) -> dict:
    """
    Calculate correlation matrices using different methods.
    """
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    corr_matrix = df[numeric_cols].corr()
    spearman_matrix = df[numeric_cols].corr(method='spearman')
    
    corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_value = corr_matrix.iloc[i, j]
            corr_pairs.append({
                'Variable 1': corr_matrix.columns[i],
                'Variable 2': corr_matrix.columns[j],
                'Correlation': corr_value,
                'Strength': abs(corr_value)
            })
    
    corr_pairs = sorted(corr_pairs, key=lambda x: x['Strength'], reverse=True)
    
    return {
        'pearson': corr_matrix,
        'spearman': spearman_matrix,
        'top_correlations': corr_pairs[:10]
    }

corr_results = correlation_analysis(df)

corr_results['pearson'].to_csv('../reports/pearson_correlation.csv')
corr_results['spearman'].to_csv('../reports/spearman_correlation.csv')

print("\nTop 5 Correlations:")
for i, pair in enumerate(corr_results['top_correlations'][:5], 1):
    print(f"{i}. {pair['Variable 1']} <-> {pair['Variable 2']}: {pair['Correlation']:.3f}")


Top 5 Correlations:
1. petal width (cm) <-> petal_area: 0.980
2. petal length (cm) <-> petal width (cm): 0.963
3. petal length (cm) <-> petal_area: 0.958
4. sepal length (cm) <-> petal length (cm): 0.874
5. sepal length (cm) <-> petal_area: 0.860
